# 14 — Random Forest bounded comparator

Chạy đúng 4 dataset multiclass, Random Forest, seed 11 và 22, 25 cây, depth 16. Đây là comparator bounded, không thay thế full RF seed study.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path.cwd()
if not (ROOT / 'research/server').is_dir():
    raise RuntimeError('Mở notebook từ thư mục gốc repository')
PYTHON = str(ROOT / '.venv-server/bin/python')
os.environ['PYTHONPATH'] = str(ROOT / 'src')
OUTPUT = ROOT / 'research/artifacts/tabular_rf_bounded_multiclass'
print({'root': str(ROOT), 'output': str(OUTPUT), 'config': 'RF seeds11/22, 25 trees, depth16'})

In [ ]:
command = [PYTHON, '-u', 'research/run_tabular_baselines.py',
    '--data', 'data/full_splits', '--runs', 'research/artifacts/full_runs_5seed',
    '--output', str(OUTPUT), '--tasks', 'multiclass', '--models', 'random_forest',
    '--seeds', '11', '22', '--n-jobs', os.environ.get('NIDS_THREADS', '16'),
    '--n-estimators', '25', '--max-depth', '16', '--prediction-cap', '100000', '--resume']
subprocess.run(command, cwd=ROOT, env=os.environ.copy(), check=True)

In [ ]:
verify = [PYTHON, '-u', 'research/validate_tabular_results.py',
    '--data', 'data/full_splits', '--runs', str(OUTPUT),
    '--output', 'research/results/tabular_rf_bounded_verification.json']
subprocess.run(verify, cwd=ROOT, env=os.environ.copy(), check=True)
import json
result = json.loads((ROOT / 'research/results/tabular_rf_bounded_verification.json').read_text())
assert result['passed'] is True and result['runs_checked'] == 8
print(result)

In [ ]:
import pandas as pd
table = pd.read_csv(OUTPUT / 'runs.csv')
display(table[['dataset', 'task', 'model', 'seed', 'test_macro_f1', 'test_weighted_f1', 'test_accuracy']])
display(table.groupby(['dataset', 'task', 'model'])['test_macro_f1'].agg(['mean', 'std']).round(4))